
# 03. Modelos baseline — dataset definitivo de contratos depurados

Este notebook treina os modelos baseline usando exclusivamente o dataset produzido pelo notebook 01 definitivo:

`dataset_modelagem_obras_snapshots_definitivo.csv`

Princípios:

- usa apenas `flag_apto_treino = 1`;
- respeita `split_modelagem` e `split_contrato_temporal` já definidos no notebook 01;
- usa `sample_weight_contrato` já calculado no dataset;
- não recria pesos nem splits;
- exclui targets, identificadores, datas finais e campos de controle das features;
- classificação é avaliada somente antes do vencimento do prazo vigente (`dias_restantes_prazo_vigente >= 0`);
- o conjunto de validação é usado para seleção de modelo/threshold;
- o teste permanece como avaliação temporal final.

Modelos baseline:

**Regressão**
- Dummy mediana
- Ridge

**Classificação**
- Dummy prior
- Regressão logística
- Regressão logística balanceada


In [1]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42


## 1. Carregamento do dataset definitivo

In [2]:

def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    PROCESSED_DIR / "dataset_modelagem_obras_snapshots_definitivo.csv",
    PROCESSED_DIR / "dataset_modelagem_obras_snapshots.csv",
    PROJECT_ROOT / "dataset_modelagem_obras_snapshots_definitivo.csv",
    PROJECT_ROOT / "dataset_modelagem_obras_snapshots.csv",
    Path("/mnt/data/data/processed/dataset_modelagem_obras_snapshots_definitivo.csv"),
    Path("/mnt/data/data/processed/dataset_modelagem_obras_snapshots.csv"),
]

DATASET_PATH = next(
    (p for p in candidate_paths if p.exists()),
    None,
)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Dataset definitivo não encontrado. "
        "Execute primeiro o notebook 01 definitivo."
    )

df = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
    encoding="utf-8-sig",
)

date_columns = [
    "data_corte",
    "data_os",
    "data_termino_original",
    "data_termino_vigente_corte",
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",
]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(
            df[column],
            errors="coerce",
        )

required = [
    "id_contrato",
    "data_corte",
    "target_dias_atraso",
    "target_atraso",
    "flag_apto_treino",
    "dias_restantes_prazo_vigente",
    "data_conclusao_definitiva",
    "data_termino_vigente_final_controle",
    "sample_weight_contrato",
    "split_contrato_temporal",
    "split_modelagem",
]

missing = [
    column
    for column in required
    if column not in df.columns
]

if missing:
    raise ValueError(
        "O dataset não possui o schema definitivo. "
        f"Colunas ausentes: {missing}"
    )

# Não recria peso ou split: eles são parte do dataset definitivo.
if df["sample_weight_contrato"].isna().any():
    raise ValueError(
        "sample_weight_contrato contém valores nulos."
    )

valid_splits = {
    "train",
    "validation",
    "test",
    "nao_supervisionado",
}

unexpected_splits = (
    set(df["split_modelagem"].dropna().unique())
    - valid_splits
)

if unexpected_splits:
    raise ValueError(
        f"Valores inesperados em split_modelagem: {unexpected_splits}"
    )

print("Dataset:", DATASET_PATH.resolve())
print("Shape:", df.shape)
print("Contratos:", df["id_contrato"].nunique())
print(
    "Snapshots aptos:",
    int(df["flag_apto_treino"].sum()),
)
print()
print("Distribuição por split:")
display(
    df.groupby("split_modelagem")
    .agg(
        snapshots=("id_contrato", "size"),
        contratos=("id_contrato", "nunique"),
    )
)


Dataset: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots_definitivo.csv
Shape: (587, 80)
Contratos: 45
Snapshots aptos: 542

Distribuição por split:


,snapshots,contratos
split_modelagem,,
nao_supervisionado,45,45
test,149,8
train,333,31
validation,60,6


## 2. Validações metodológicas antes do treino

In [3]:

model_data = df[
    df["flag_apto_treino"].eq(1)
    & df["split_modelagem"].isin(
        [
            "train",
            "validation",
            "test",
        ]
    )
].copy()

if model_data.empty:
    raise ValueError(
        "Não há snapshots supervisionados para modelagem."
    )

# Cada contrato pode pertencer a apenas um split.
contract_splits = (
    model_data
    .groupby("id_contrato")["split_modelagem"]
    .nunique()
)

assert contract_splits.max() == 1, (
    "Há contratos presentes em mais de um split."
)

# A foto supervisionada deve ocorrer antes da conclusão.
assert (
    model_data["data_corte"]
    < model_data["data_conclusao_definitiva"]
).all(), (
    "Há snapshots supervisionados na/apos a conclusão."
)

# Target consistente.
target_recalc = (
    model_data["data_conclusao_definitiva"]
    - model_data[
        "data_termino_vigente_final_controle"
    ]
).dt.days.clip(lower=0)

assert np.allclose(
    target_recalc.astype(float),
    model_data["target_dias_atraso"].astype(float),
), "target_dias_atraso inconsistente."

assert np.array_equal(
    (
        model_data["target_dias_atraso"] > 0
    ).astype(int).to_numpy(),
    model_data["target_atraso"].astype(int).to_numpy(),
), "target_atraso inconsistente."

print(
    "Contratos supervisionados:",
    model_data["id_contrato"].nunique(),
)
print(
    "Snapshots supervisionados:",
    len(model_data),
)
print(
    "Taxa de atraso por snapshot:",
    f"{model_data['target_atraso'].mean():.1%}",
)

contract_targets = (
    model_data
    .sort_values("data_corte")
    .drop_duplicates("id_contrato")[
        ["id_contrato", "target_atraso", "target_dias_atraso"]
    ]
)

print(
    "Taxa de atraso por contrato:",
    f"{contract_targets['target_atraso'].mean():.1%}",
)


Contratos supervisionados: 45
Snapshots supervisionados: 542
Taxa de atraso por snapshot: 38.4%
Taxa de atraso por contrato: 37.8%



## 3. Seleção explícita de features

As datas absolutas e todos os controles finais são excluídos.

As features candidatas abaixo são todas produzidas point-in-time pelo notebook 01.


In [4]:

FORBIDDEN = {
    # identificadores
    "id_contrato",

    # datas brutas / auditoria
    "data_corte",
    "data_os",
    "data_termino_original",
    "data_termino_vigente_corte",
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",

    # controle final
    "status_final_controle",
    "diferenca_vigente_original_final_controle",

    # targets
    "target_dias_atraso",
    "target_atraso",

    # elegibilidade / pesos / split
    "flag_apto_treino",
    "sample_weight_contrato_raw",
    "sample_weight_contrato",
    "split_contrato_temporal",
    "split_modelagem",
}

numeric_candidates = [
    "prazo_original_dias",
    "dias_desde_os",
    "dias_restantes_prazo_vigente",
    "prazo_vigente_total_dias",
    "pct_prazo_consumido",
    "pct_prazo_restante",

    "qtd_aditivos_prazo",
    "dias_aditados_acumulados",
    "pct_dias_aditados_sobre_prazo_original",
    "dias_desde_ultimo_aditivo_prazo",
    "qtd_aditivos_valor",
    "dias_desde_ultimo_aditivo_valor",

    "qtd_parcelas_previstas_ate_corte",
    "qtd_parcelas_concluidas_ate_corte",
    "pct_parcelas_concluidas_vs_previstas",
    "atraso_medio_parcelas_dias",
    "atraso_mediano_parcelas_dias",
    "atraso_maximo_parcelas_dias",
    "qtd_parcelas_atrasadas",
    "pct_parcelas_atrasadas",
    "dias_desde_ultima_parcela_concluida",
    "delta_atraso_90d_vs_historico",

    "qtd_liberacoes",
    "valor_liberado_acumulado",
    "atraso_medio_liberacao_dias",
    "atraso_maximo_liberacao_dias",
    "dias_desde_ultima_liberacao",

    "gap_prazo_execucao",
    "pressao_tempo_execucao",

    "teve_parcela_concluida",
    "teve_liberacao",
    "teve_aditivo_prazo",
    "teve_aditivo_valor",

    "recent_qtd_parcelas_concluidas_30d",
    "recent_qtd_parcelas_atrasadas_30d",
    "recent_pct_parcelas_atrasadas_30d",
    "recent_atraso_medio_parcelas_30d",
    "recent_atraso_maximo_parcelas_30d",
    "recent_qtd_liberacoes_30d",
    "recent_valor_liberado_30d",
    "recent_atraso_medio_liberacao_30d",

    "recent_qtd_parcelas_concluidas_60d",
    "recent_qtd_parcelas_atrasadas_60d",
    "recent_pct_parcelas_atrasadas_60d",
    "recent_atraso_medio_parcelas_60d",
    "recent_atraso_maximo_parcelas_60d",
    "recent_qtd_liberacoes_60d",
    "recent_valor_liberado_60d",
    "recent_atraso_medio_liberacao_60d",

    "recent_qtd_parcelas_concluidas_90d",
    "recent_qtd_parcelas_atrasadas_90d",
    "recent_pct_parcelas_atrasadas_90d",
    "recent_atraso_medio_parcelas_90d",
    "recent_atraso_maximo_parcelas_90d",
    "recent_qtd_liberacoes_90d",
    "recent_valor_liberado_90d",
    "recent_atraso_medio_liberacao_90d",

    "ano_corte",
    "mes_corte",
    "mes_sin",
    "mes_cos",
]

numeric_features = [
    c
    for c in numeric_candidates
    if c in df.columns
]

if "classificacao_contrato_agrupada" in df.columns:
    categorical_features = [
        "classificacao_contrato_agrupada"
    ]
elif "classificacao_contrato" in df.columns:
    categorical_features = [
        "classificacao_contrato"
    ]
else:
    categorical_features = []

features = (
    numeric_features
    + categorical_features
)

if not features:
    raise ValueError(
        "Nenhuma feature foi selecionada."
    )

assert not (
    set(features) & FORBIDDEN
), "Feature proibida foi incluída."

print(
    "Features numéricas:",
    len(numeric_features),
)
print(
    "Features categóricas:",
    len(categorical_features),
)
print(
    "Total de features:",
    len(features),
)

print("\nFeature categórica:", categorical_features)


Features numéricas: 61
Features categóricas: 1
Total de features: 62

Feature categórica: ['classificacao_contrato']


## 4. Preprocessamento

In [5]:

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)

preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
)


def split_xy(data, split):
    part = data[
        data["split_modelagem"].eq(split)
    ].copy()

    X = part[features]
    y_reg = (
        part["target_dias_atraso"]
        .astype(float)
    )
    y_clf = (
        part["target_atraso"]
        .astype(int)
    )
    w = (
        part["sample_weight_contrato"]
        .astype(float)
    )

    return (
        part,
        X,
        y_reg,
        y_clf,
        w,
    )



# Parte A — Regressão

A regressão usa todos os snapshots supervisionados.

As previsões são truncadas em zero porque o target não pode ser negativo.


In [6]:

(
    reg_train,
    Xr_train,
    yr_train,
    _,
    wr_train,
) = split_xy(
    model_data,
    "train",
)

(
    reg_val,
    Xr_val,
    yr_val,
    _,
    wr_val,
) = split_xy(
    model_data,
    "validation",
)

(
    reg_test,
    Xr_test,
    yr_test,
    _,
    wr_test,
) = split_xy(
    model_data,
    "test",
)

for name, part in [
    ("train", reg_train),
    ("validation", reg_val),
    ("test", reg_test),
]:
    if part.empty:
        raise ValueError(
            f"Split de regressão vazio: {name}"
        )

regression_models = {
    "dummy_mediana": Pipeline(
        [
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                DummyRegressor(
                    strategy="median",
                ),
            ),
        ]
    ),
    "ridge": Pipeline(
        [
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                Ridge(
                    alpha=10.0,
                ),
            ),
        ]
    ),
}


def regression_metrics(
    y_true,
    y_pred,
    sample_weight,
):
    y_pred = np.maximum(
        0,
        np.asarray(y_pred),
    )

    error = (
        np.asarray(y_pred)
        - np.asarray(y_true)
    )

    return {
        "mae_ponderado": mean_absolute_error(
            y_true,
            y_pred,
            sample_weight=sample_weight,
        ),
        "medae": median_absolute_error(
            y_true,
            y_pred,
        ),
        "rmse_ponderado": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
                sample_weight=sample_weight,
            )
        ),
        "r2_ponderado": r2_score(
            y_true,
            y_pred,
            sample_weight=sample_weight,
        ),
        "pct_erro_abs_ate_30d": np.average(
            np.abs(error) <= 30,
            weights=sample_weight,
        ),
        "erro_assinado_medio": np.average(
            error,
            weights=sample_weight,
        ),
    }


reg_models_fitted = {}
reg_results = []

for name, model in regression_models.items():
    model.fit(
        Xr_train,
        yr_train,
        model__sample_weight=wr_train,
    )

    reg_models_fitted[name] = model

    for split_name, X, y, w in [
        (
            "validation",
            Xr_val,
            yr_val,
            wr_val,
        ),
        (
            "test",
            Xr_test,
            yr_test,
            wr_test,
        ),
    ]:
        pred = np.maximum(
            0,
            model.predict(X),
        )

        result = {
            "modelo": name,
            "split": split_name,
            **regression_metrics(
                y,
                pred,
                w,
            ),
        }

        reg_results.append(result)

reg_results = pd.DataFrame(reg_results)

display(
    reg_results.sort_values(
        [
            "split",
            "mae_ponderado",
        ]
    )
)

validation_reg = reg_results[
    reg_results["split"].eq(
        "validation"
    )
].sort_values(
    "mae_ponderado"
)

BEST_REGRESSION_MODEL = (
    validation_reg.iloc[0]["modelo"]
)

print(
    "Melhor baseline de regressão "
    "pela MAE ponderada de validação:",
    BEST_REGRESSION_MODEL,
)


,modelo,split,mae_ponderado,medae,rmse_ponderado,r2_ponderado,pct_erro_abs_ate_30d,erro_assinado_medio
1,dummy_mediana,test,21.8750,0.0000,54.3519,-0.1933,0.8750,-21.8750
3,ridge,test,73.4026,86.0542,86.6684,-2.0342,0.2876,40.3830
0,dummy_mediana,validation,14.8333,14.0000,31.1475,-0.2933,0.8333,-14.8333
2,ridge,validation,19.8309,22.2603,25.5949,0.1267,0.7531,16.6985


Melhor baseline de regressão pela MAE ponderada de validação: dummy_mediana



## Diagnóstico da regressão por contrato

Além das métricas por snapshot, calculamos uma avaliação com uma observação por contrato no teste: o snapshot mais próximo de 90 dias do prazo vigente.


In [7]:

best_reg = reg_models_fitted[
    BEST_REGRESSION_MODEL
]

reg_test_eval = reg_test.copy()
reg_test_eval["pred_dias_atraso"] = np.maximum(
    0,
    best_reg.predict(Xr_test),
)

HORIZONTE_DIAS = 90

reg_fixed_horizon = reg_test_eval[
    reg_test_eval[
        "dias_restantes_prazo_vigente"
    ].ge(0)
].copy()

reg_fixed_horizon["dist_horizonte"] = (
    reg_fixed_horizon[
        "dias_restantes_prazo_vigente"
    ]
    - HORIZONTE_DIAS
).abs()

reg_fixed_horizon = (
    reg_fixed_horizon
    .sort_values(
        [
            "id_contrato",
            "dist_horizonte",
            "data_corte",
        ]
    )
    .groupby(
        "id_contrato",
        as_index=False,
    )
    .first()
)

if len(reg_fixed_horizon):
    print(
        f"Contratos no teste avaliados próximos de {HORIZONTE_DIAS} dias:",
        reg_fixed_horizon["id_contrato"].nunique(),
    )
    print(
        "MAE por contrato:",
        mean_absolute_error(
            reg_fixed_horizon[
                "target_dias_atraso"
            ],
            reg_fixed_horizon[
                "pred_dias_atraso"
            ],
        ),
    )

display(
    reg_fixed_horizon[
        [
            "id_contrato",
            "data_corte",
            "dias_restantes_prazo_vigente",
            "target_dias_atraso",
            "pred_dias_atraso",
        ]
    ]
)


Contratos no teste avaliados próximos de 90 dias: 8
MAE por contrato: 21.875


,id_contrato,data_corte,dias_restantes_prazo_vigente,target_dias_atraso,pred_dias_atraso
0,173,2024-07-31,80,0.0000,0.0000
1,201,2024-11-30,102,0.0000,0.0000
2,254,2025-05-31,83,0.0000,0.0000
3,283,2024-11-30,97,0.0000,0.0000
4,287,2025-03-31,84,0.0000,0.0000
5,317,2024-11-30,84,23.0000,0.0000
6,342,2025-07-18,60,0.0000,0.0000
7,377,2025-04-29,90,152.0000,0.0000



# Parte B — Classificação

Para classificação, utilizamos apenas snapshots em que o prazo vigente ainda não venceu:

`dias_restantes_prazo_vigente >= 0`

Isso evita transformar o problema em uma identificação trivial de contratos cujo prazo já passou.


In [8]:

clf_data = model_data[
    model_data[
        "dias_restantes_prazo_vigente"
    ].ge(0)
].copy()

assert (
    clf_data[
        "dias_restantes_prazo_vigente"
    ].ge(0)
).all()

(
    clf_train,
    Xc_train,
    _,
    yc_train,
    wc_train,
) = split_xy(
    clf_data,
    "train",
)

(
    clf_val,
    Xc_val,
    _,
    yc_val,
    wc_val,
) = split_xy(
    clf_data,
    "validation",
)

(
    clf_test,
    Xc_test,
    _,
    yc_test,
    wc_test,
) = split_xy(
    clf_data,
    "test",
)

for name, y in [
    ("train", yc_train),
    ("validation", yc_val),
    ("test", yc_test),
]:
    print(
        name,
        "| snapshots:",
        len(y),
        "| taxa atraso:",
        f"{y.mean():.1%}",
        "| classes:",
        sorted(y.unique().tolist()),
    )

if yc_train.nunique() < 2:
    raise ValueError(
        "Treino de classificação possui apenas uma classe."
    )

classification_models = {
    "dummy_prior": Pipeline(
        [
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                DummyClassifier(
                    strategy="prior",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "logistic": Pipeline(
        [
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                LogisticRegression(
                    C=1.0,
                    max_iter=5000,
                    solver="liblinear",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "logistic_balanced": Pipeline(
        [
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                LogisticRegression(
                    C=1.0,
                    max_iter=5000,
                    solver="liblinear",
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}


def safe_auc(y_true, prob, sample_weight=None):
    if pd.Series(y_true).nunique() < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        prob,
        sample_weight=sample_weight,
    )


def safe_ap(y_true, prob, sample_weight=None):
    if pd.Series(y_true).nunique() < 2:
        return np.nan

    return average_precision_score(
        y_true,
        prob,
        sample_weight=sample_weight,
    )


def classification_metrics(
    y_true,
    prob,
    threshold,
    sample_weight,
):
    pred = (
        np.asarray(prob) >= threshold
    ).astype(int)

    return {
        "threshold": threshold,
        "accuracy_ponderada": accuracy_score(
            y_true,
            pred,
            sample_weight=sample_weight,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            pred,
            sample_weight=sample_weight,
        ),
        "precision": precision_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "brier": brier_score_loss(
            y_true,
            prob,
            sample_weight=sample_weight,
        ),
        "roc_auc": safe_auc(
            y_true,
            prob,
            sample_weight,
        ),
        "pr_auc": safe_ap(
            y_true,
            prob,
            sample_weight,
        ),
    }


clf_models_fitted = {}
clf_results_05 = []

for name, model in classification_models.items():
    model.fit(
        Xc_train,
        yc_train,
        model__sample_weight=wc_train,
    )

    clf_models_fitted[name] = model

    for split_name, X, y, w in [
        (
            "validation",
            Xc_val,
            yc_val,
            wc_val,
        ),
        (
            "test",
            Xc_test,
            yc_test,
            wc_test,
        ),
    ]:
        prob = (
            model.predict_proba(X)[:, 1]
        )

        clf_results_05.append({
            "modelo": name,
            "split": split_name,
            **classification_metrics(
                y,
                prob,
                threshold=0.50,
                sample_weight=w,
            ),
        })

clf_results_05 = pd.DataFrame(
    clf_results_05
)

display(
    clf_results_05.sort_values(
        [
            "split",
            "roc_auc",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


train | snapshots: 299 | taxa atraso: 40.5% | classes: [0, 1]
validation | snapshots: 57 | taxa atraso: 54.4% | classes: [0, 1]
test | snapshots: 137 | taxa atraso: 11.7% | classes: [0, 1]


,modelo,split,threshold,accuracy_ponderada,balanced_accuracy,precision,recall,f1,brier,roc_auc,pr_auc
3,logistic,test,0.5000,0.4443,0.6215,0.2132,0.8854,0.3436,0.4186,0.5524,0.1848
5,logistic_balanced,test,0.5000,0.4339,0.6153,0.2100,0.8854,0.3395,0.4563,0.5341,0.1743
1,dummy_prior,test,0.5000,0.8357,0.5000,0.0000,0.0000,0.0000,0.1862,0.5000,0.1643
0,dummy_prior,validation,0.5000,0.6803,0.5000,0.0000,0.0000,0.0000,0.2218,0.5000,0.3197
4,logistic_balanced,validation,0.5000,0.2649,0.3651,0.2487,0.6430,0.3587,0.5244,0.4083,0.2865
2,logistic,validation,0.5000,0.2735,0.3658,0.2471,0.6217,0.3537,0.4898,0.3959,0.2790



## Seleção do classificador e threshold

O modelo é selecionado pela ROC-AUC na validação. Depois, o threshold é escolhido na validação para maximizar a balanced accuracy.

O conjunto de teste não participa dessa escolha.


In [9]:

validation_clf = (
    clf_results_05[
        clf_results_05["split"].eq(
            "validation"
        )
    ]
    .copy()
)

# Se a validação tiver duas classes, usa ROC-AUC.
# Caso contrário, usa balanced accuracy.
if validation_clf[
    "roc_auc"
].notna().any():
    validation_clf = validation_clf.sort_values(
        [
            "roc_auc",
            "balanced_accuracy",
        ],
        ascending=[
            False,
            False,
        ],
    )
else:
    validation_clf = validation_clf.sort_values(
        "balanced_accuracy",
        ascending=False,
    )

BEST_CLASSIFICATION_MODEL = (
    validation_clf.iloc[0]["modelo"]
)

best_clf = clf_models_fitted[
    BEST_CLASSIFICATION_MODEL
]

prob_val = best_clf.predict_proba(
    Xc_val
)[:, 1]

threshold_grid = np.arange(
    0.05,
    0.951,
    0.01,
)

threshold_rows = []

for threshold in threshold_grid:
    metrics = classification_metrics(
        yc_val,
        prob_val,
        threshold=threshold,
        sample_weight=wc_val,
    )

    threshold_rows.append(metrics)

threshold_results = pd.DataFrame(
    threshold_rows
)

BEST_THRESHOLD = float(
    threshold_results.sort_values(
        [
            "balanced_accuracy",
            "f1",
        ],
        ascending=[
            False,
            False,
        ],
    ).iloc[0]["threshold"]
)

print(
    "Melhor classificador:",
    BEST_CLASSIFICATION_MODEL,
)
print(
    "Threshold escolhido na validação:",
    round(BEST_THRESHOLD, 2),
)

prob_test = best_clf.predict_proba(
    Xc_test
)[:, 1]

final_classification = pd.DataFrame([
    {
        "split": "validation",
        **classification_metrics(
            yc_val,
            prob_val,
            threshold=BEST_THRESHOLD,
            sample_weight=wc_val,
        ),
    },
    {
        "split": "test",
        **classification_metrics(
            yc_test,
            prob_test,
            threshold=BEST_THRESHOLD,
            sample_weight=wc_test,
        ),
    },
])

display(final_classification)

pred_test = (
    prob_test >= BEST_THRESHOLD
).astype(int)

print("Matriz de confusão — teste:")
print(
    confusion_matrix(
        yc_test,
        pred_test,
    )
)


Melhor classificador: dummy_prior
Threshold escolhido na validação: 0.05


,split,threshold,accuracy_ponderada,balanced_accuracy,precision,recall,f1,brier,roc_auc,pr_auc
0,validation,0.0500,0.3197,0.5000,0.3197,1.0000,0.4845,0.2218,0.5000,0.3197
1,test,0.0500,0.1643,0.5000,0.1643,1.0000,0.2822,0.1862,0.5000,0.1643


Matriz de confusão — teste:
[[  0 121]
 [  0  16]]



## Avaliação da classificação por contrato em horizonte aproximado de 90 dias


In [10]:

clf_test_eval = clf_test.copy()
clf_test_eval["prob_atraso"] = prob_test
clf_test_eval["pred_atraso"] = pred_test

HORIZONTE_DIAS = 90

fixed_horizon = clf_test_eval.copy()

fixed_horizon["dist_horizonte"] = (
    fixed_horizon[
        "dias_restantes_prazo_vigente"
    ]
    - HORIZONTE_DIAS
).abs()

fixed_horizon = (
    fixed_horizon
    .sort_values(
        [
            "id_contrato",
            "dist_horizonte",
            "data_corte",
        ]
    )
    .groupby(
        "id_contrato",
        as_index=False,
    )
    .first()
)

display(
    fixed_horizon[
        [
            "id_contrato",
            "data_corte",
            "dias_restantes_prazo_vigente",
            "target_atraso",
            "prob_atraso",
            "pred_atraso",
        ]
    ]
)

if fixed_horizon["target_atraso"].nunique() > 1:
    print(
        "Balanced accuracy por contrato (~90d):",
        balanced_accuracy_score(
            fixed_horizon["target_atraso"],
            fixed_horizon["pred_atraso"],
        ),
    )
    print(
        "ROC-AUC por contrato (~90d):",
        roc_auc_score(
            fixed_horizon["target_atraso"],
            fixed_horizon["prob_atraso"],
        ),
    )


,id_contrato,data_corte,dias_restantes_prazo_vigente,target_atraso,prob_atraso,pred_atraso
0,173,2024-07-31,80,0,0.3853,1
1,201,2024-11-30,102,0,0.3853,1
2,254,2025-05-31,83,0,0.3853,1
3,283,2024-11-30,97,0,0.3853,1
4,287,2025-03-31,84,0,0.3853,1
5,317,2024-11-30,84,1,0.3853,1
6,342,2025-07-18,60,0,0.3853,1
7,377,2025-04-29,90,1,0.3853,1


Balanced accuracy por contrato (~90d): 0.5
ROC-AUC por contrato (~90d): 0.5


## 5. Sanity checks finais contra leakage

In [11]:

assert not (
    set(features) & FORBIDDEN
), (
    "Há coluna proibida em X."
)

assert (
    clf_data[
        "dias_restantes_prazo_vigente"
    ].ge(0)
).all()

assert (
    model_data
    .groupby("id_contrato")[
        "split_modelagem"
    ]
    .nunique()
    .max()
    == 1
)

# As datas finais não podem ter sido convertidas em feature.
forbidden_dates = {
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",
}

assert not (
    set(features) & forbidden_dates
)

print(
    "Sanity checks concluídos com sucesso."
)


Sanity checks concluídos com sucesso.


## 6. Exportação das métricas

In [12]:

REG_METRICS_PATH = (
    REPORTS_DIR
    / "03_baseline_regressao_definitivo.csv"
)

CLF_05_METRICS_PATH = (
    REPORTS_DIR
    / "03_baseline_classificacao_threshold_050_definitivo.csv"
)

CLF_FINAL_METRICS_PATH = (
    REPORTS_DIR
    / "03_baseline_classificacao_final_definitivo.csv"
)

THRESHOLD_PATH = (
    REPORTS_DIR
    / "03_baseline_thresholds_validacao_definitivo.csv"
)

reg_results.to_csv(
    REG_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

clf_results_05.to_csv(
    CLF_05_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

final_classification.to_csv(
    CLF_FINAL_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

threshold_results.to_csv(
    THRESHOLD_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Métricas salvas em:")
print(REG_METRICS_PATH.resolve())
print(CLF_05_METRICS_PATH.resolve())
print(CLF_FINAL_METRICS_PATH.resolve())
print(THRESHOLD_PATH.resolve())

print()
print("Melhor baseline regressão:", BEST_REGRESSION_MODEL)
print("Melhor baseline classificação:", BEST_CLASSIFICATION_MODEL)
print("Threshold:", round(BEST_THRESHOLD, 2))


Métricas salvas em:
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\reports\models\03_baseline_regressao_definitivo.csv
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\reports\models\03_baseline_classificacao_threshold_050_definitivo.csv
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\reports\models\03_baseline_classificacao_final_definitivo.csv
C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\reports\models\03_baseline_thresholds_validacao_definitivo.csv

Melhor baseline regressão: dummy_mediana
Melhor baseline classificação: dummy_prior
Threshold: 0.05
